In [1]:
pip install --upgrade langchain langchain-core langchain-gigachat langgraph langchain-community

   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 17.9 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 10.6 MB/s  0:00:00

   ------------ ---------------------------  6/19 [dataclasses-json]
   -------------- -------------------------  7/19 [langsmith]
   -------------- -------------------------  7/19 [langsmith]
   -------------- -------------------------  7/19 [langsmith]
   ---------------- -----------------------  8/19 [langgraph-sdk]
   ------------------ ---------------------  9/19 [langchain-core]
   ------------------ ---------------------  9/19 [langchain-core]
   ------------------ ---------------------  9/19 [langchain-core]
   ------------------ ---------------------  9/19 [langchain-core]
   --------------------- ------------------ 10/19 [gigachat]
   --------------------------- ------------ 13/19 [langchain-gi

In [2]:
import os
from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, START, END
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import ChatPromptTemplate

model = GigaChat(
    credentials="MDE5YzIzYmQtMWUzNS03M2FlLWE3NDYtZWFmZDc1N2I4OTVjOmRhYjg3MDU5LTc5NTctNGZjZS04YTc4LWQ0M2JlOGMzOWMwOA==",
    scope="GIGACHAT_API_PERS",
    model="GigaChat-Pro",
    verify_ssl_certs=False,
)

In [3]:
class RobotState(TypedDict):
    command: str                    
    envt: str        
    battery: int                
    complexity: str            
    action_plan: List[str]          
    obstacles: bool         
    current_action: str             
    execution_log: List[str]        
    final_status: str               

In [5]:
def dispatcher_node(state: RobotState) -> Dict[str, Any]:
    print("Анализ полученной команды...")
    print(f"Команда: '{state['command']}'")
    print(f"Уровень заряда: {state['battery']}%")
    if state['battery'] < 20:
        print("Низкий заряд батареи. Выполнение задачи невозможно.")
        return {
            "complexity": "aborted",
            "final_status": "ABORTED_LOW_BATTERY",
            "execution_log": ["Задача отклонена: критически низкий заряд."]
        }  
    prompt = ChatPromptTemplate.from_template(
        """Ты - анализатор задач для робота-курьера.     
Команда пользователя: {command}
Оцени сложность этой задачи. Ответь ТОЛЬКО одним словом:
- "complex" - если задача требует принятия решений, поиска, обхода препятствий, проверки условий, доставки нескольких предметов
- "simple" - если задача прямолинейна: принести один предмет, поехать в одно место
"""
    )
    chain = prompt | model
    response = chain.invoke({"command": state['command']})
    complexity = response.content.strip().lower()
    return {
        "complexity": complexity,
        "execution_log": [f"Команда принята. Сложность: {complexity}"]
    }

In [6]:
def simple_planner_node(state: RobotState) -> Dict[str, Any]:
    print("Идет построение простого маршрута...")
    
    prompt = ChatPromptTemplate.from_template(
        """Ты - система простого планирования для робота.        
Контекст: {environment}
Команда: {command}
Дай КОРОТКИЙ план (3-5 шагов) без дополнительного анализа.
Доступные действия: navigate_to, pickup, deliver, return_to_base.
Верни ТОЛЬКО список действий, каждое с новой строки."""
    )
    chain = prompt | model
    response = chain.invoke({
        "environment": state['envt'],
        "command": state['command']
    })  
    action_plan = [line.strip() for line in response.content.split('\n') if line.strip()]
    print(f"План действий:")
    for action in action_plan: print(f" - {action}")
    return {
        "action_plan": action_plan,
        "execution_log": ["Базовый маршрут построен."]
    }

def complex_planner_node(state: RobotState) -> Dict[str, Any]:
    print("Идет построение сложного маршрута...")
    
    prompt = ChatPromptTemplate.from_template(
        """Ты - система углубленного планирования для робота.        
Контекст окружения: {environment}
Полученная команда: {command}
Уровень заряда: {battery}%
Составь последовательность действий для робота в виде списка команд.
Доступные команды робота:
- scan_environment
- navigate_to(location)
- wait_for_confirmation
- pickup_item
- avoid_obstacle
- deliver_item
- return_to_base
Верни ТОЛЬКО список команд, каждая с новой строки, без нумерации и дополнительного текста."""
    )
    chain = prompt | model
    response = chain.invoke({
        "environment": state['envt'],
        "command": state['command'],
        "battery": state['battery']
    })
    action_plan = [line.strip() for line in response.content.split('\n') if line.strip()]
    print(f"План действий:")
    for i, action in enumerate(action_plan, 1): print(f"   {i}. {action}")
    return {
        "action_plan": action_plan,
        "execution_log": ["Детальный план построен."]
    }
    

In [7]:
def obstacles_node(state: RobotState) -> Dict[str, Any]:
    print("Идет поиск препятствий...")
    prompt = ChatPromptTemplate.from_template(
        "В этом описании есть препятствие для робота? Ответь только 'да' или 'нет'. Описание: {env}"
    )
    chain = prompt | model
    response = chain.invoke({"env": state['envt']})
    obstacles = "да" in response.content.lower()
    if obstacles:
        print("Обнаружено препятствие. Маршрут будет скорректирован.")
        updated_plan = ["avoid_obstacle"] + state['action_plan']
        log_entry = "Препятствие обнаружено. План скорректирован."
    else:
        print("Путь свободен.")
        updated_plan = state['action_plan']
        log_entry = "Препятствий не обнаружено."
    return {
        "obstacles": obstacles,
        "action_plan": updated_plan,
        "execution_log": state.get("execution_log", []) + [log_entry]
    }

In [8]:
def executor_node(state: RobotState) -> Dict[str, Any]:
    print("Идет выполнение задачи...")
    execution_log = state.get("execution_log", [])
    for i, action in enumerate(state['action_plan'], 1):
        print(f"Выполняется шаг {i}/{len(state['action_plan'])}: {action}")
        execution_log.append(f"[Действие {i}] {action}")
        battery_after = state['battery'] - (i * 2)
        if battery_after < 15 and action != "emergency_return_to_base":
            print(f" Низкий заряд ({battery_after}%). Выполнение задачи невозможно")
            execution_log.append("[Исполнитель] Выполнение прервано из-за низкого заряда.")
            return {
                "current_action": "emergency_stop",
                "execution_log": execution_log,
                "final_status": "INTERRUPTED_LOW_BATTERY"
            }
    print("Все действия выполнены успешно!")
    execution_log.append("Задача завершена успешно.")
    return {
        "current_action": "completed",
        "execution_log": execution_log,
        "final_status": "SUCCESS"
    }

In [9]:
def report_generator_node(state: RobotState) -> Dict[str, Any]:
    print("Отчет о миссии:")
    print(f"Исходная команда: {state['command']}")
    print(f"Сложность задачи: {state['complexity']}")
    print(f"Препятствия: {'Обнаружены' if state.get('obstacles') else 'Отсутствуют'}")
    print(f"Итоговый статус: {state['final_status']}")
    print(f"Выполнено шагов: {len(state['action_plan'])}")
    return {}

In [10]:
def route_by_complexity(state: RobotState) -> str:
    complexity = state['complexity']
    if state.get('final_status') == "ABORTED_LOW_BATTERY":
        return "ReportGenerator"
    if complexity == "complex":
        return "ComplexPlanner"
    else:
        return "SimplePlanner"

In [14]:
workflow = StateGraph(RobotState)

workflow.add_node("Dispatcher", dispatcher_node)
workflow.add_node("SimplePlanner", simple_planner_node)
workflow.add_node("ComplexPlanner", complex_planner_node)
workflow.add_node("ObstacleHandler", obstacles_node)
workflow.add_node("Executor", executor_node)
workflow.add_node("ReportGenerator", report_generator_node)

workflow.add_edge(START, "Dispatcher")

workflow.add_conditional_edges(
    "Dispatcher",
    route_by_complexity,
    {
        "SimplePlanner": "SimplePlanner",
        "ComplexPlanner": "ComplexPlanner",
        "ReportGenerator": "ReportGenerator"
    }
)

workflow.add_edge("SimplePlanner", "ObstacleHandler")
workflow.add_edge("ComplexPlanner", "ObstacleHandler")
workflow.add_edge("ObstacleHandler", "Executor")
workflow.add_edge("Executor", "ReportGenerator")
workflow.add_edge("ReportGenerator", END)

app = workflow.compile()

In [16]:
if __name__ == "__main__":
    print("СЦЕНАРИЙ 1: ПРОСТАЯ ЗАДАЧА")
    
    initial_state_1: RobotState = {
        "command": "Принеси кофе из кухни",
        "envt": "Коридор свободен, кухня справа, освещение нормальное",
        "battery": 85,
        "complexity": "",
        "action_plan": [],
        "obstacles": False,
        "current_action": "",
        "execution_log": [],
        "final_status": ""
    }
    
    result_1 = app.invoke(initial_state_1)
    
    print("СЦЕНАРИЙ 2: СЛОЖНАЯ ЗАДАЧА С ПРЕПЯТСТВИЕМ")
    
    initial_state_2: RobotState = {
        "command": "Найди синюю папку с документами и доставь её в переговорную, если там нет совещания",
        "envt": "Коридор частично заблокирован коробками, переговорная в конце коридора, требуется проверка занятости",
        "battery": 45,
        "complexity": "",
        "action_plan": [],
        "obstacles": False,
        "current_action": "",
        "execution_log": [],
        "final_status": ""
    }
    
    result_2 = app.invoke(initial_state_2)
    
    print("СЦЕНАРИЙ 3: КРИТИЧЕСКИ НИЗКИЙ ЗАРЯД БАТАРЕИ")
    
    initial_state_3: RobotState = {
        "command": "Принеси воду",
        "envt": "Коридор свободен",
        "battery": 12,
        "complexity": "",
        "action_plan": [],
        "obstacles": False,
        "current_action": "",
        "execution_log": [],
        "final_status": ""
    }
    
    result_3 = app.invoke(initial_state_3)
    
    print("ВСЕ СЦЕНАРИИ ВЫПОЛНЕНЫ")

СЦЕНАРИЙ 1: ПРОСТАЯ ЗАДАЧА

ДИСПЕТЧЕР: Анализирую полученную команду...
Команда: 'Принеси кофе из кухни'
Уровень заряда: 85%
Задача простая.
Идет построение простого маршрута...
План действий:
 - navigate_to kitchen
 - pickup coffee
 - deliver coffee
 - return_to_base
Идет поиск препятствий...
Путь свободен.
Идет выполнение задачи...
Выполняется шаг 1/4: navigate_to kitchen
Выполняется шаг 2/4: pickup coffee
Выполняется шаг 3/4: deliver coffee
Выполняется шаг 4/4: return_to_base
Все действия выполнены успешно!
Отчет о миссии:
Исходная команда: Принеси кофе из кухни
Сложность задачи: simple
Препятствия: Отсутствуют
Итоговый статус: SUCCESS
Выполнено шагов: 4
СЦЕНАРИЙ 2: СЛОЖНАЯ ЗАДАЧА С ПРЕПЯТСТВИЕМ

ДИСПЕТЧЕР: Анализирую полученную команду...
Команда: 'Найди синюю папку с документами и доставь её в переговорную, если там нет совещания'
Уровень заряда: 45%
Задача сложная и требует детального планирования.
Идет построение сложного маршрута...
План действий:
   1. scan_environment
   2. a